# Relic Scarcity Math Breakdown

This notebook documents the full math behind:
- Material scarcity multipliers
- Signal data farming cadence
- Corrupted signal data conversion
- Derived relic tier weights

It uses the project file `data/relic_costs.json` as the source of truth and recomputes downstream values transparently.

In [25]:
from pathlib import Path
import json

DATA_PATH = Path('../data/relic_costs.json')
assert DATA_PATH.exists(), f'Missing data file: {DATA_PATH.resolve()}'

with DATA_PATH.open('r', encoding='utf-8') as f:
    relic_data = json.load(f)

relic_data['version'], relic_data['last_updated']

('1.5', '2026-05-16')

## 1) Current Material Scarcity Multipliers

These are the active scarcity multipliers consumed by the downstream scoring pipeline.

In [26]:
material_scarcity = relic_data['material_scarcity'].copy()
description = material_scarcity.pop('description')
print('Description:')
print(description)
print()

for name, weight in sorted(material_scarcity.items(), key=lambda kv: kv[1]):
    print(f'{name:28s} -> {weight:8.2f}x')

Description:
Scarcity multipliers based on token availability plus shared-energy signal-data bundle optimization. Baseline=1.0 (carbonite_circuit_board). Base scenario uses 400 daily cantina energy with weekly shop, TB, and Assault Battle injections and is stress-tested with low/base/high cadence sensitivity. Corrupted signal data uses the lower-cost source between direct Mk III purchase and salvage conversion (1 per 145 scrap).

carbonite_circuit_board      ->     1.00x
bronzium_wiring              ->     1.14x
chromium_transistor          ->     1.43x
signal_data_incomplete       ->     1.49x
signal_data_fragmented       ->     1.50x
signal_data_flawed           ->     1.50x
aurodium_heatsink            ->     1.57x
electrium_conductor          ->     3.65x
zinbiddle_card               ->     4.86x
signal_data_corrupted        ->    21.75x
aeromagnifier                ->    37.71x
impulse_detector             ->    49.97x
gyrda_keypad                 ->   110.31x
droid_brain         

## 2) Signal Data Farming Model

Assumptions used in recent calibration (now parameterized by scenario):
- Cantina energy per day
- Weekly shop bursts
- Monthly TB injections
- Assault Battles per month and event rewards

This section now computes two supply views:
- Independent max per signal (upper bound)
- Shared-energy optimized bundle farming (more realistic)

It also prints low/base/high sensitivity to show how cadence assumptions move results.

In [20]:
from random import Random

days_per_month = 30

nodes = {
    '9B': {'energy': 20, 'signal_data_fragmented': 1.27, 'signal_data_incomplete': 0.58},
    '9D': {'energy': 20, 'signal_data_fragmented': 1.29, 'signal_data_flawed': 0.41},
    '9F': {'energy': 20, 'signal_data_incomplete': 0.85, 'signal_data_flawed': 0.50},
    '8G': {'energy': 16, 'signal_data_flawed': 1.36},
    '8F': {'energy': 16, 'signal_data_incomplete': 1.02},
    '8C': {'energy': 16, 'signal_data_fragmented': 0.62},
}

signal_types = ['signal_data_fragmented', 'signal_data_incomplete', 'signal_data_flawed']
tier_costs = relic_data['tier_costs']
required_signal = {s: 0 for s in signal_types}
for tier in tier_costs.values():
    for s in signal_types:
        required_signal[s] += tier.get(s, 0)

best_per_energy = {
    signal: max(node.get(signal, 0.0) / node['energy'] for node in nodes.values())
    for signal in signal_types
}

def monthly_extras(config):
    weeks_per_month = config['days_per_month'] / 7
    return {
        'signal_data_fragmented': config['weekly_fragmented'] * weeks_per_month + config['tb_fragmented'] + config['ab_fragmented'] * config['ab_events_per_month'],
        'signal_data_incomplete': config['weekly_incomplete'] * weeks_per_month + config['tb_incomplete'] + config['ab_incomplete'] * config['ab_events_per_month'],
        'signal_data_flawed': config['ab_flawed'] * config['ab_events_per_month'],
    }

def independent_supply(config):
    extras = monthly_extras(config)
    cantina = {
        s: best_per_energy[s] * config['cantina_energy_per_day'] * config['days_per_month']
        for s in signal_types
    }
    total = {s: cantina[s] + extras[s] for s in signal_types}
    return cantina, extras, total

def shared_supply_for_allocation(config, allocation):
    total_energy = config['cantina_energy_per_day'] * config['days_per_month']
    cantina = {s: 0.0 for s in signal_types}
    for node_name, share in allocation.items():
        node = nodes[node_name]
        for s in signal_types:
            cantina[s] += share * total_energy * (node.get(s, 0.0) / node['energy'])
    extras = monthly_extras(config)
    total = {s: cantina[s] + extras[s] for s in signal_types}
    return cantina, extras, total

def optimize_bundle_allocation(config, samples=10000, seed=11):
    rng = Random(seed)
    node_names = list(nodes.keys())
    best_bundle_rate = -1.0
    best_allocation = None
    best_totals = None

    for _ in range(samples):
        draws = [rng.gammavariate(1.0, 1.0) for _ in node_names]
        draw_sum = sum(draws)
        allocation = {name: draws[i] / draw_sum for i, name in enumerate(node_names)}
        _, _, totals = shared_supply_for_allocation(config, allocation)
        bundle_rate = min(totals[s] / required_signal[s] for s in signal_types)
        if bundle_rate > best_bundle_rate:
            best_bundle_rate = bundle_rate
            best_allocation = allocation
            best_totals = totals

    return best_bundle_rate, best_allocation, best_totals

scenario_configs = {
    'low': {
        'days_per_month': days_per_month,
        'cantina_energy_per_day': 350,
        'ab_events_per_month': 6,
        'weekly_fragmented': 40,
        'weekly_incomplete': 35,
        'tb_fragmented': 20,
        'tb_incomplete': 20,
        'ab_fragmented': 10,
        'ab_incomplete': 6,
        'ab_flawed': 6,
    },
    'base': {
        'days_per_month': days_per_month,
        'cantina_energy_per_day': 400,
        'ab_events_per_month': 7,
        'weekly_fragmented': 45,
        'weekly_incomplete': 40,
        'tb_fragmented': 20,
        'tb_incomplete': 20,
        'ab_fragmented': 10,
        'ab_incomplete': 6,
        'ab_flawed': 6,
    },
    'high': {
        'days_per_month': days_per_month,
        'cantina_energy_per_day': 450,
        'ab_events_per_month': 8,
        'weekly_fragmented': 50,
        'weekly_incomplete': 45,
        'tb_fragmented': 20,
        'tb_incomplete': 20,
        'ab_fragmented': 10,
        'ab_incomplete': 6,
        'ab_flawed': 6,
    },
}

base_config = scenario_configs['base']
cantina_energy_per_day = base_config['cantina_energy_per_day']
ab_events_per_month = base_config['ab_events_per_month']

monthly_cantina, monthly_extras_values, monthly_total_independent = independent_supply(base_config)
bundle_rate_shared, best_allocation, monthly_total_shared = optimize_bundle_allocation(base_config)
monthly_total = monthly_total_shared

print('Best per-energy yield (independent upper bound):')
for s in signal_types:
    print(f'  {s:24s} {best_per_energy[s]:.5f}')

print('\nBase scenario monthly totals - independent upper bound:')
for s in signal_types:
    print(f'  {s:24s} {monthly_total_independent[s]:8.2f}')

print('\nBase scenario monthly totals - shared-energy optimized allocation:')
for s in signal_types:
    print(f'  {s:24s} {monthly_total_shared[s]:8.2f}')

print('\nBest shared-energy allocation (% cantina energy):')
for name, share in sorted(best_allocation.items(), key=lambda kv: kv[1], reverse=True):
    print(f'  {name:3s}: {share * 100:6.2f}%')

print('\nSensitivity (days per full R10 signal bundle):')
print(f"{'Scenario':10s} {'Independent':>12s} {'Shared':>12s}")
for label, config in scenario_configs.items():
    _, _, totals_ind = independent_supply(config)
    rate_ind = min(totals_ind[s] / required_signal[s] for s in signal_types)
    days_ind = config['days_per_month'] / rate_ind

    rate_shared, _, _ = optimize_bundle_allocation(config, samples=4000, seed=17)
    days_shared = config['days_per_month'] / rate_shared

    print(f"{label:10s} {days_ind:12.3f} {days_shared:12.3f}")

Best per-energy yield (independent upper bound):
  signal_data_fragmented   0.06450
  signal_data_incomplete   0.06375
  signal_data_flawed       0.08500

Base scenario monthly totals - independent upper bound:
  signal_data_fragmented    1056.86
  signal_data_incomplete     998.43
  signal_data_flawed        1062.00

Base scenario monthly totals - shared-energy optimized allocation:
  signal_data_fragmented     356.78
  signal_data_incomplete     526.16
  signal_data_flawed         595.90

Best shared-energy allocation (% cantina energy):
  8G :  51.01%
  8F :  27.44%
  9F :  10.89%
  9B :   7.83%
  8C :   2.46%
  9D :   0.36%

Sensitivity (days per full R10 signal bundle):
Scenario    Independent       Shared
low               7.108       12.968
base              6.215       11.391
high              5.521       10.156


## 3) Signal Scarcity From R10 Demand

Compute total signal requirements across tiers 1-10, then estimate how many days each signal takes to farm one full R10 bundle.

Relative scarcity among signal types is normalized against fragmented.

In [21]:
required_signal = {s: 0 for s in signal_types}
for tier in tier_costs.values():
    for s in signal_types:
        required_signal[s] += tier.get(s, 0)

days_independent = {
    s: (required_signal[s] / monthly_total_independent[s]) * days_per_month
    for s in signal_types
}
days_shared = {
    s: (required_signal[s] / monthly_total_shared[s]) * days_per_month
    for s in signal_types
}

baseline_ind = days_independent['signal_data_fragmented']
baseline_shared = days_shared['signal_data_fragmented']
relative_ind = {s: days_independent[s] / baseline_ind for s in signal_types}
relative_shared = {s: days_shared[s] / baseline_shared for s in signal_types}

# Use shared-energy model as the conservative default for downstream interpretation.
days_for_bundle = days_shared
signal_relative = relative_shared

print('R10 signal requirements:')
for s in signal_types:
    print(f'  {s:24s} {required_signal[s]:5d}')

print('\nDays to farm each full-signal requirement bundle (independent upper bound):')
for s in signal_types:
    print(f'  {s:24s} {days_independent[s]:8.3f} days')

print('\nRelative scarcity vs fragmented (independent upper bound):')
for s in signal_types:
    print(f'  {s:24s} {relative_ind[s]:8.3f}x')

print('\nDays to farm each full-signal requirement bundle (shared-energy optimized):')
for s in signal_types:
    print(f'  {s:24s} {days_shared[s]:8.3f} days')

print('\nRelative scarcity vs fragmented (shared-energy optimized):')
for s in signal_types:
    print(f'  {s:24s} {relative_shared[s]:8.3f}x')

R10 signal requirements:
  signal_data_fragmented     135
  signal_data_incomplete     195
  signal_data_flawed         220

Days to farm each full-signal requirement bundle (independent upper bound):
  signal_data_fragmented      3.832 days
  signal_data_incomplete      5.859 days
  signal_data_flawed          6.215 days

Relative scarcity vs fragmented (independent upper bound):
  signal_data_fragmented      1.000x
  signal_data_incomplete      1.529x
  signal_data_flawed          1.622x

Days to farm each full-signal requirement bundle (shared-energy optimized):
  signal_data_fragmented     11.351 days
  signal_data_incomplete     11.118 days
  signal_data_flawed         11.076 days

Relative scarcity vs fragmented (shared-energy optimized):
  signal_data_fragmented      1.000x
  signal_data_incomplete      0.979x
  signal_data_flawed          0.976x


## 4) Corrupted Signal Data: Direct vs Conversion

Corrupted has two supply paths:
- Direct Mk III purchase path
- Conversion path through salvage scrap

Rule: use the lower effective scarcity (cheaper path).

With the current calibrated signal multipliers:
- fragmented = 1.50, incomplete = 1.486, flawed = 1.50
- conversion route lands at 21.75x via flawed
- direct path is set to the same 21.75x effective scarcity

In [27]:
# Conversion math from provided rates
# 1 corrupted = 145 scrap
# scrap per signal: frag=6, incomplete=8, flawed=10
frag_weight = relic_data['material_scarcity']['signal_data_fragmented']
inc_weight = relic_data['material_scarcity']['signal_data_incomplete']
flawed_weight = relic_data['material_scarcity']['signal_data_flawed']

corr_via_frag = (145 / 6) * frag_weight
corr_via_inc = (145 / 8) * inc_weight
corr_via_flawed = (145 / 10) * flawed_weight
best_conversion = min(corr_via_frag, corr_via_inc, corr_via_flawed)
direct_weight = relic_data['material_scarcity']['signal_data_corrupted']
effective = min(best_conversion, direct_weight)

print('Corrupted via fragmented:', round(corr_via_frag, 3))
print('Corrupted via incomplete:', round(corr_via_inc, 3))
print('Corrupted via flawed    :', round(corr_via_flawed, 3))
print('Best conversion path    :', round(best_conversion, 3))
print('Current direct path     :', round(direct_weight, 3))
print('Effective scarcity      :', round(effective, 3))

Corrupted via fragmented: 36.25
Corrupted via incomplete: 26.934
Corrupted via flawed    : 21.75
Best conversion path    : 21.75
Current direct path     : 21.75
Effective scarcity      : 21.75


## 5) Recompute Tier Weights From Scarcity

This mirrors downstream logic in the app:
- Sum each tier's required materials weighted by `material_scarcity`
- Normalize by tier 1 to get relative weights

In [28]:
scarcity = {k: v for k, v in relic_data['material_scarcity'].items() if k != 'description'}

required_materials = set()
for tier_str, tier in relic_data['tier_costs'].items():
    if not tier_str.isdigit():
        continue
    for mat in tier:
        if mat not in ('credits', 'description'):
            required_materials.add(mat)

missing_scarcity = sorted(required_materials - set(scarcity))
if missing_scarcity:
    raise KeyError(
        'Missing material_scarcity entries for: ' + ', '.join(missing_scarcity)
    )

raw_tier_weight = {}
for tier_str, tier in relic_data['tier_costs'].items():
    if not tier_str.isdigit():
        continue
    t = int(tier_str)
    weighted = 0.0
    for mat, amt in tier.items():
        if mat in ('credits', 'description'):
            continue
        weighted += amt * scarcity[mat]
    raw_tier_weight[t] = weighted

base = raw_tier_weight[1]
normalized = {t: raw_tier_weight[t] / base for t in sorted(raw_tier_weight)}

print('Tier -> normalized scarcity weight')
for t in sorted(normalized):
    print(f'  {t:2d} -> {normalized[t]:8.2f}x')

print('\nKey ratios:')
print('  R10 vs R3:', round(normalized[10] / normalized[3], 2), 'x')
print('  R10 vs R8:', round(normalized[10] / normalized[8], 2), 'x')

Tier -> normalized scarcity weight
   1 ->     1.00x
   2 ->     2.45x
   3 ->     3.91x
   4 ->     5.00x
   5 ->     5.99x
   6 ->     7.65x
   7 ->     8.89x
   8 ->    52.96x
   9 ->   161.58x
  10 ->   221.49x

Key ratios:
  R10 vs R3: 56.62 x
  R10 vs R8: 4.18 x


## 6) Compare Against JSON Fallback Relative Weights

The `relative_weights` field is now maintained as a fallback snapshot recomputed from current `material_scarcity` values.
The live scoring path still computes scarcity-driven relic costs directly when `material_scarcity` is present.

Current JSON fallback set (tier-normalized):
- R1: 1.00
- R2: 2.45
- R3: 3.91
- R4: 5.00
- R5: 5.99
- R6: 7.65
- R7: 8.89
- R8: 52.96
- R9: 161.58
- R10: 221.49

In [29]:
legacy = {int(k): float(v) for k, v in relic_data['relative_weights'].items() if str(k).isdigit()}

print(f"{'Tier':>4}  {'Legacy':>10}  {'Scarcity':>10}  {'Scarcity/Legacy':>16}")
for t in range(1, 11):
    l = legacy[t]
    s = normalized[t]
    print(f"{t:4d}  {l:10.2f}  {s:10.2f}  {s/l:16.2f}")

Tier      Legacy    Scarcity   Scarcity/Legacy
   1        1.00        1.00              1.00
   2        2.45        2.45              1.00
   3        3.91        3.91              1.00
   4        5.00        5.00              1.00
   5        5.99        5.99              1.00
   6        7.65        7.65              1.00
   7        8.89        8.89              1.00
   8       52.96       52.96              1.00
   9      161.58      161.58              1.00
  10      221.49      221.49              1.00


## 7) Optional Tuning Knobs

To tune assumptions, edit values in `scenario_configs` in Cell 6 and rerun Cells 6 through 14.
Useful knobs include:
- `cantina_energy_per_day`
- `ab_events_per_month`
- weekly shop burst quantities
- TB monthly injections
- AB per-event rewards

This sensitivity pass helps calibrate scarcity behavior to your guild and account cadence.